# PsychoPy Hold-Release Task Analysis: Early vs Final Release Latency
This notebook demonstrates a simple behavioural analysis workflow using output from a PsychoPy hold-release task.

The aim is to extract key response and release-timing measures from the raw PsychoPy CSV file, clean the data, identify early releases, and summarise final release latency across trials.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

## Load the PsychoPy output

The raw data file is a CSV exported by PsychoPy. Each row represents one trial, and the columns include key press, key release, and timing information.

In [ ]:
df = pd.read_csv("PSYCHOPY TASKS/errorcancdemo/data/hold_release_clean_output.csv")
df.head()
df.columns
display(df)

## Rename columns
The full PsychoPy output contains many timing and metadata columns. For this analysis, I selected the columns related to trial identity, initial key press, early release, and final release.

In [ ]:
clean_df = df[["cond","dur","msg","press_start.keys","press_start.rt","early_release.keys","early_release.rt","final_release.keys","final_release.rt", "thisN", "thisTrialN"]].copy()
display(clean_df)

# Add trial numbers

In [ ]:
clean_df["trials"] = range(1, len(clean_df) + 1)
display(clean_df)

## Select useful columns

The full PsychoPy output contains many timing and metadata columns. For this analysis, I selected the columns related to trial identity, initial key press, early release, and final release.

In [ ]:
clean_df = clean_df.rename(columns={ "press_start.keys" : "press_start", "press_start.rt" : "press_rt", "early_release.keys" : "early_release_key", "early_release.rt" : "early_release_rt", "final_release.keys" : "final_release_keys", "final_release.rt" : "final_release_rt"})
display(clean_df)

## Identify early releases

An early release occurs when the participant releases the key during the hold period, before the final release signal. I created a Boolean column to mark whether an early release occurred on each trial.

In [ ]:
clean_df["early_release_made"] = clean_df["early_release_key"].notna()
display(clean_df)
early_trials = clean_df[clean_df["early_release_made"] == True].copy()
normal_trials = clean_df[clean_df["early_release_made"] == False].copy()

display(early_trials)
display(normal_trials)

# Count early releases

In [ ]:
clean_df["early_release_made"].sum()

## Summarise release latencies
Early-release RT measures how quickly the participant released during the hold period. Final-release RT measures how quickly the participant released after the final release signal on normal trials.

In [ ]:
mean_final_release_rt = normal_trials["final_release_rt"].mean()
mean_final_release_rt


early_trials["early_release_rt"].describe()
normal_trials["final_release_rt"].describe()

## Plot release latencies across trials

In [ ]:
plt.plot(early_trials["trials"], early_trials["early_release_rt"], marker="o")
plt.xlabel("Trial")
plt.ylabel("Early release RT (s)")
plt.title("Early release latency across trials")
plt.show()

plt.plot(normal_trials["trials"], normal_trials["final_release_rt"], marker="o")
plt.xlabel("Trial")
plt.ylabel("Final release RT (s)")
plt.title("Final release latency across normal trials")
plt.show()

## Brief interpretation

This notebook analyses output from the v5 hold-release PsychoPy demo. In this version, if the participant releases during the KEEP HOLDING period, the task records an early release and skips the final RELEASE NOW routine. If the participant keeps holding, the task proceeds to the final release signal and records final release latency.

The analysis separates early-release trials from normal final-release trials. Early-release RT summarises how quickly the participant released during the hold period, while final-release RT summarises how quickly the participant released after the release signal on normal trials.

This is a small demonstration dataset, but it shows a cleaner behavioural-analysis workflow than the earlier prototype because premature releases are no longer mixed with final-release responses.